In [2]:
import os
import itertools
import multiprocessing as mp
import shutil

from concurrent.futures import ProcessPoolExecutor
from collections import defaultdict

import numpy as np
import pandas as pd
import supervision as sv

from trackers import SORTTracker
from trackers.eval import evaluate_mot_sequences


DATA_ROOT = "."
SPORTSMOT_DET_ROOT = os.path.join(DATA_ROOT, "sportsmot_yolox_dets")
SPORTSMOT_GT_ROOT = os.path.join(DATA_ROOT, "TrackEval", "data", "gt", "sportsmot")

MIN_BOX_AREA = 10
VERTICAL_RATIO_THRESH = 1.6


def get_yolox_detections(frame_id, det_list):
    dets_per_frame = [d for d in det_list if d.split(",")[0] == str(frame_id)]
    dets = []
    for line in dets_per_frame:
        det = line.split(",")
        x1, y1, x2, y2 = float(det[1]), float(det[2]), float(det[3]), float(det[4])
        conf = float(det[5])
        dets.append([x1, y1, x2, y2, conf])
    return dets


def build_dets_index(det_list):
    dets_by_frame = defaultdict(list)
    for line in det_list:
        frame_id = int(line.split(",")[0])
        dets_by_frame[frame_id].append(line)
    return dets_by_frame


def get_detections_from_dict(frame_id, dets_by_frame):
    dets = []
    for line in dets_by_frame[frame_id]:
        det = line.split(",")
        x1, y1, x2, y2 = float(det[1]), float(det[2]), float(det[3]), float(det[4])
        conf = float(det[5])
        dets.append([x1, y1, x2, y2, conf])
    return dets

In [3]:
def run_sort_sportsmot_once(
    split: str,
    lost_track_buffer: int,
    frame_rate: float,
    track_activation_threshold: float,
    minimum_consecutive_frames: int,
    minimum_iou_threshold: float,
):
    assert split in {"val", "test"}

    tracker = SORTTracker(
        lost_track_buffer=lost_track_buffer,
        frame_rate=frame_rate,
        track_activation_threshold=track_activation_threshold,
        minimum_consecutive_frames=minimum_consecutive_frames,
        minimum_iou_threshold=minimum_iou_threshold,
    )

    det_root = os.path.join(SPORTSMOT_DET_ROOT, split)
    outputs_root = "SORT_outputs_sportsmot"
    os.makedirs(outputs_root, exist_ok=True)

    save_dir = os.path.join(
        outputs_root,
        f"{split}_L{lost_track_buffer}_F{frame_rate}_A{track_activation_threshold}_"
        f"C{minimum_consecutive_frames}_I{minimum_iou_threshold}",
    )
    os.makedirs(save_dir, exist_ok=True)

    for seq in sorted(os.listdir(det_root)):
        if not seq.endswith(".txt"):
            continue
        tracker.reset()
        seq_name = seq.split(".")[0]
        with open(os.path.join(det_root, seq), "r") as f_det:
            det_list = f_det.readlines()
            dets_by_frame = build_dets_index(det_list)
        last_frame = int(det_list[-1].split(",")[0])
        output_lines = []
        for frame_id in range(1, last_frame + 1):
            raw_dets = get_detections_from_dict(frame_id, dets_by_frame)
            if raw_dets:
                raw_dets = np.array(raw_dets)
                dets = sv.Detections(xyxy=raw_dets[:, :4], confidence=raw_dets[:, 4])
            else:
                dets = sv.Detections.empty()
            dets = tracker.update(detections=dets)
            for tid, (left, top, right, bottom) in zip(dets.tracker_id, dets.xyxy):
                if tid == -1:
                    continue
                width = right - left
                height = bottom - top
                vertical = width / max(height, 1e-6) > VERTICAL_RATIO_THRESH
                if width * height > MIN_BOX_AREA and not vertical:
                    output_lines.append(
                        f"{frame_id},{int(tid)},{round(left,1)},{round(top,1)},{round(width,1)},{round(height,1)},-1,-1,-1,-1\n"
                    )
        save_path = os.path.join(save_dir, seq_name + ".txt")
        with open(save_path, "w") as f:
            f.writelines(output_lines)

    HOTA = IDF1 = MOTA = None
    gt_dir = os.path.join(SPORTSMOT_GT_ROOT, split)
    try:
        result = evaluate_mot_sequences(gt_dir=gt_dir, tracker_dir=save_dir, metrics=["CLEAR", "HOTA", "Identity"])
        MOTA = result.to_dict()["aggregate"]["CLEAR"]["MOTA"]
        HOTA = result.to_dict()["aggregate"]["HOTA"]["HOTA"]
        IDF1 = result.to_dict()["aggregate"]["Identity"]["IDF1"]
        print(f"split={split} | L:{lost_track_buffer}, F:{frame_rate}, A:{track_activation_threshold}, C:{minimum_consecutive_frames}, I:{minimum_iou_threshold} -> HOTA: {HOTA:.3f}, IDF1: {IDF1:.3f}, MOTA: {MOTA:.3f}")
    except Exception as e:
        print(f"split={split} | evaluation FAILED ({repr(e)}). Results in {save_dir}")

    return {
        "split": split,
        "lost_track_buffer": lost_track_buffer,
        "frame_rate": frame_rate,
        "track_activation_threshold": track_activation_threshold,
        "minimum_consecutive_frames": minimum_consecutive_frames,
        "minimum_iou_threshold": minimum_iou_threshold,
        "HOTA": HOTA, "IDF1": IDF1, "MOTA": MOTA,
        "output_dir": save_dir,
    }

In [ ]:
FRAME_RATE = 30.0  # fixed, not tuned
lost_track_buffer_candidates = [10, 30, 60]
track_activation_threshold_candidates = [0.25, 0.5, 0.75, 0.9]
minimum_consecutive_frames_candidates = [2, 3, 5]
minimum_iou_threshold_candidates = [0.05, 0.1, 0.3, 0.5, 0.7]

all_candidate_lists = [
    lost_track_buffer_candidates,
    track_activation_threshold_candidates,
    minimum_consecutive_frames_candidates,
    minimum_iou_threshold_candidates,
]
parameter_combinations = list(itertools.product(*all_candidate_lists))
print(f"Total SORT parameter combinations: {len(parameter_combinations)}")

Total SORT parameter combinations: 180


In [7]:
def tune_sort_on_val_parallel(parameter_combinations):
    num_workers = os.cpu_count()
    print(f"Running {len(parameter_combinations)} SORT combinations on val with {num_workers} workers")
    ctx = mp.get_context("fork")
    all_results = []
    with ProcessPoolExecutor(max_workers=num_workers, mp_context=ctx) as executor:
        futures = []
        for i, comb in enumerate(parameter_combinations):
            L, A, C, I = comb
            futures.append(executor.submit(run_sort_sportsmot_once, "val", L, FRAME_RATE, A, C, I))
        for i, f in enumerate(futures):
            try:
                all_results.append(f.result())
                print(f"[{i+1}/{len(parameter_combinations)}] val combination finished.")
            except Exception as e:
                print(f"[{i+1}/{len(parameter_combinations)}] FAILED: {repr(e)}")
    df = pd.DataFrame(all_results)
    if not df.empty:
        df.to_csv("sort_sportsmot_val_tuning.csv", index=False)
    return df

sort_sportsmot_val_tuning_df = tune_sort_on_val_parallel(parameter_combinations)

if not sort_sportsmot_val_tuning_df.empty and "HOTA" in sort_sportsmot_val_tuning_df.columns and not sort_sportsmot_val_tuning_df["HOTA"].isna().all():
    best_idx = sort_sportsmot_val_tuning_df["HOTA"].idxmax()
    best_row = sort_sportsmot_val_tuning_df.loc[best_idx]
    best_params = dict(
        lost_track_buffer=int(best_row["lost_track_buffer"]),
        frame_rate=FRAME_RATE,
        track_activation_threshold=float(best_row["track_activation_threshold"]),
        minimum_consecutive_frames=int(best_row["minimum_consecutive_frames"]),
        minimum_iou_threshold=float(best_row["minimum_iou_threshold"]),
    )
    print("Best validation row by HOTA:", best_row)
    print("Best params:", best_params)
else:
    print("No HOTA in results; set best_params manually below.")
    best_params = None

Running 180 SORT combinations on val with 10 workers
split=val | L:10, F:30.0, A:0.25, C:3, I:0.7 -> HOTA: 0.351, IDF1: 0.280, MOTA: 0.789
split=val | L:10, F:30.0, A:0.25, C:2, I:0.7 -> HOTA: 0.355, IDF1: 0.277, MOTA: 0.820
split=val | L:10, F:30.0, A:0.25, C:2, I:0.5 -> HOTA: 0.707, IDF1: 0.669, MOTA: 0.951
split=val | L:10, F:30.0, A:0.25, C:2, I:0.1 -> HOTA: 0.797, IDF1: 0.787, MOTA: 0.981
split=val | L:10, F:30.0, A:0.25, C:3, I:0.05 -> HOTA: 0.796, IDF1: 0.787, MOTA: 0.979
split=val | L:10, F:30.0, A:0.25, C:2, I:0.3 -> HOTA: 0.787, IDF1: 0.773, MOTA: 0.977
split=val | L:10, F:30.0, A:0.25, C:2, I:0.05 -> HOTA: 0.798, IDF1: 0.788, MOTA: 0.982
split=val | L:10, F:30.0, A:0.25, C:3, I:0.5 -> HOTA: 0.706, IDF1: 0.670, MOTA: 0.945
split=val | L:10, F:30.0, A:0.25, C:3, I:0.3 -> HOTA: 0.785, IDF1: 0.773, MOTA: 0.973
[1/180] val combination finished.
[2/180] val combination finished.
[3/180] val combination finished.
[4/180] val combination finished.
[5/180] val combination finished.
[

## Final test run and submission zip

In [ ]:
if best_params is None:
    best_params = dict(lost_track_buffer=30, frame_rate=FRAME_RATE, track_activation_threshold=0.5, minimum_consecutive_frames=3, minimum_iou_threshold=0.3)

print("Running SORT on SportsMOT test with:", best_params)
test_result = run_sort_sportsmot_once("test", **best_params)

submission_dir = test_result["output_dir"]
submission_zip_base = os.path.basename(submission_dir)
shutil.make_archive("sort_"+submission_zip_base, "zip", submission_dir)
print("Created submission archive:", submission_zip_base + ".zip")

Running SORT on SportsMOT test with: {'lost_track_buffer': 60, 'frame_rate': 30.0, 'track_activation_threshold': 0.9, 'minimum_consecutive_frames': 2, 'minimum_iou_threshold': 0.05}
split=test | evaluation FAILED (FileNotFoundError('Ground truth directory not found: TrackEval/data/gt/sportsmot/test')). Results in SORT_outputs_sportsmot/test_L60_F30.0_A0.9_C2_I0.05
Created submission archive: test_L60_F30.0_A0.9_C2_I0.05.zip
